# Experiment: conditioning-ladder VAE round-trip diagnosis

This Colab notebook reproduces the failed clean VAE preflight for the 17 fixed conditions from the conditioning-overfitting ladder. It answers one narrow question: **which condition fails, and is the failure caused by 30×30 padding, batch execution, or neither?**

It loads only the frozen FLUX.2 VAE. It does not load NVARC or the FLUX transformer, does not train, does not change the palette, and does not regenerate any conditions.

Success criteria:

- reproduce the exact production `30×30` encode/decode path for all 17 conditions;
- identify every non-exact condition with its decoded grid, PAD-boundary state, cell mismatches, and nearest-palette margins;
- compare production batching against one-by-one and all-at-once execution;
- compare the same 3×3 targets without padding using the compact `3×3` VAE path;
- save one compact contact sheet and one machine-readable report to Google Drive.


## Colab prerequisites

Use a GPU runtime. The notebook expects these existing Drive files and does not download model weights or ARC data:

```text
MyDrive/diffusion-arc/
├── models/flux2-klein-base-4b/vae/
│   ├── config.json
│   └── diffusion_pytorch_model.safetensors
└── data/raw/arc-agi-2/training/0d3d703e.json
```

The notebook fetches only the pinned Diffusion-ARC source commit. If the GitHub repository is private, add `GITHUB_TOKEN` as a Colab secret; its value is never printed or written to Drive.


In [ ]:
!nvidia-smi


## 1. Fix the exact source, artifacts, and runtime identity

`SOURCE_REVISION` is deliberately the commit used by the failed ladder run. Keeping it fixed makes the condition registry, renderer, decoder, and VAE wrapper identical to the original preflight.


In [ ]:
from __future__ import annotations

import base64
from datetime import datetime, timezone
import hashlib
import importlib
import importlib.metadata
import json
import os
from pathlib import Path
import random
import runpy
import shutil
import subprocess
import sys
import time

import numpy as np
import pandas as pd
import torch
from google.colab import drive, userdata
from IPython.display import display

REPOSITORY_URL = 'https://github.com/joshuakauling/diffusion-arc.git'
SOURCE_REVISION = 'd7cc6632ad6bde7bef177a959670fcb40ad499ec'
TASK_ID = '0d3d703e'
EXPECTED_TASK_SHA256 = '16bc1695ae651124ea07b7c1d08248a02500391ed1a2a6bd059be8ab6851e0c8'
EXPECTED_VAE_SHA256 = 'ca70d2202afe6415bdbcb8793ba8cd99fd159cfe6192381504d6c4d3036e0f04'
SEED = 20260826

DRIVE_MOUNT = Path('/content/drive')
DRIVE_ROOT = DRIVE_MOUNT / 'MyDrive' / 'diffusion-arc'
VAE_SOURCE_PATH = DRIVE_ROOT / 'models' / 'flux2-klein-base-4b' / 'vae'
ARC_TASK_PATH = DRIVE_ROOT / 'data' / 'raw' / 'arc-agi-2' / 'training' / f'{TASK_ID}.json'
LOCAL_VAE_PATH = Path('/content/flux2-klein-base-4b-vae-roundtrip-diagnosis')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT_DIR = DRIVE_ROOT / 'outputs' / 'conditioning-ladder-vae-roundtrip-diagnosis' / RUN_ID
SOURCE_DIR = Path('/content') / f'diffusion-arc-roundtrip-source-{RUN_ID}'

assert len(SOURCE_REVISION) == 40 and all(character in '0123456789abcdef' for character in SOURCE_REVISION)
drive.mount(str(DRIVE_MOUNT), force_remount=False)
os.environ.update({
    'HF_HUB_OFFLINE': '1',
    'TRANSFORMERS_OFFLINE': '1',
    'HF_HUB_DISABLE_TELEMETRY': '1',
    'TOKENIZERS_PARALLELISM': 'false',
})
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU.'
DEVICE = torch.device('cuda:0')
torch.cuda.set_device(DEVICE)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
random.seed(SEED)
np.random.seed(SEED)
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)

print({
    'run_id': RUN_ID,
    'gpu': torch.cuda.get_device_properties(DEVICE).name,
    'gpu_memory_GiB': round(torch.cuda.get_device_properties(DEVICE).total_memory / 2**30, 1),
    'source_revision': SOURCE_REVISION,
    'task_path': str(ARC_TASK_PATH),
    'output_dir': str(OUTPUT_DIR),
})


## 2. Fetch the exact source and install its pinned runtime

The small source checkout is detached at the failed-run commit. The notebook runs the repository test suite before any model is loaded.


In [ ]:
def run_command(label: str, command: list[str], *, cwd: Path | None = None, env: dict[str, str] | None = None) -> str:
    completed = subprocess.run(
        command, cwd=cwd, env=env, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, check=False,
    )
    if completed.returncode != 0:
        tail = '\n'.join(completed.stdout.splitlines()[-60:])
        if tail:
            print(tail)
        raise RuntimeError(f'{label} failed with exit code {completed.returncode}')
    return completed.stdout.strip()

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

SOURCE_DIR.mkdir(parents=True, exist_ok=False)
run_command('git init', ['git', 'init'], cwd=SOURCE_DIR)
run_command('add origin', ['git', 'remote', 'add', 'origin', REPOSITORY_URL], cwd=SOURCE_DIR)
try:
    github_token = userdata.get('GITHUB_TOKEN')
except Exception:
    github_token = os.environ.get('GITHUB_TOKEN')
github_token = (github_token or '').strip()
git_environment = os.environ.copy()
git_environment['GIT_TERMINAL_PROMPT'] = '0'
if github_token:
    credential = base64.b64encode(f'x-access-token:{github_token}'.encode('utf-8')).decode('ascii')
    git_environment.update({
        'GIT_CONFIG_COUNT': '1',
        'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader',
        'GIT_CONFIG_VALUE_0': f'AUTHORIZATION: basic {credential}',
    })
try:
    run_command('fetch source', ['git', 'fetch', '--depth=1', 'origin', SOURCE_REVISION], cwd=SOURCE_DIR, env=git_environment)
    run_command('checkout source', ['git', 'checkout', '--detach', 'FETCH_HEAD'], cwd=SOURCE_DIR)
finally:
    git_environment.clear()
    github_token = None
    if 'credential' in locals():
        del credential
RESOLVED_SOURCE_COMMIT = run_command('resolve source', ['git', 'rev-parse', 'HEAD'], cwd=SOURCE_DIR)
assert RESOLVED_SOURCE_COMMIT == SOURCE_REVISION

def installed_version(name: str) -> str | None:
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None

PINNED_PACKAGES = {
    'accelerate': '1.12.0',
    'diffusers': '0.39.0',
    'einops': '0.8.1',
    'Pillow': '11.3.0',
    'safetensors': '0.8.0',
    'tqdm': '4.67.1',
    'transformers': '4.56.1',
}
requirements = [
    f'{name}=={version}' for name, version in PINNED_PACKAGES.items()
    if installed_version(name) != version
]
if requirements:
    run_command('install pinned dependencies', [sys.executable, '-m', 'pip', 'install', '--quiet', '--no-input', '--no-cache-dir', *requirements])
# Colab currently uses Python 3.13, which is deliberately outside the
# project's packaged-support range.  This VAE-only diagnostic therefore
# imports the pinned source checkout directly instead of installing it.
SOURCE_PATH = str((SOURCE_DIR / 'src').resolve())
if SOURCE_PATH not in sys.path:
    sys.path.insert(0, SOURCE_PATH)
importlib.invalidate_caches()
print(f'Using pinned Diffusion-ARC source directly: {SOURCE_PATH}')

test_environment = os.environ.copy()
test_environment.update({'PYTHONDONTWRITEBYTECODE': '1', 'PYTHONPATH': SOURCE_PATH})
tests_started = time.perf_counter()
tests = subprocess.run([sys.executable, '-B', '-m', 'unittest', 'discover', '-s', 'tests'], cwd=SOURCE_DIR, env=test_environment, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, check=False)
print('\n'.join(tests.stdout.splitlines()[-35:]))
if tests.returncode != 0:
    raise RuntimeError(f'Repository unit tests returned {tests.returncode}')
print({'resolved_source_commit': RESOLVED_SOURCE_COMMIT, 'unit_tests_seconds': round(time.perf_counter() - tests_started, 1)})


## 3. Validate the VAE and reconstruct the fixed 17-condition target panel

The following cells use the production `FluxVaeEncoder` and `FluxVaeDecoder` wrappers, including their FLUX latent packing and normalization. The targets are built from `CONDITION_SPECS` in the pinned runner, not recreated manually.


In [ ]:
required_vae_files = ('config.json', 'diffusion_pytorch_model.safetensors')
missing = [name for name in required_vae_files if not (VAE_SOURCE_PATH / name).is_file()]
if missing:
    raise FileNotFoundError(f'Incomplete Drive VAE directory: {missing}')
if not ARC_TASK_PATH.is_file():
    raise FileNotFoundError(f'Copy the single ARC task to Drive: {ARC_TASK_PATH}')
if not LOCAL_VAE_PATH.exists():
    shutil.copytree(VAE_SOURCE_PATH, LOCAL_VAE_PATH)
missing_local = [name for name in required_vae_files if not (LOCAL_VAE_PATH / name).is_file()]
if missing_local:
    raise RuntimeError(f'Incomplete local VAE copy: {missing_local}')
vae_sha256 = sha256_file(LOCAL_VAE_PATH / 'diffusion_pytorch_model.safetensors')
task_sha256 = sha256_file(ARC_TASK_PATH)
assert vae_sha256 == EXPECTED_VAE_SHA256, (EXPECTED_VAE_SHA256, vae_sha256)
assert task_sha256 == EXPECTED_TASK_SHA256, (EXPECTED_TASK_SHA256, task_sha256)

from diffusers import AutoencoderKLFlux2
from diffusion_arc.arc_data import (
    PAD_CLASS, TARGET_PALETTE_ID, TARGET_PALETTE_RGB, apply_training_augmentation,
    classify_arc_cells, decode_arc_grids, decode_fixed_arc_grids, load_training_task,
    render_grid_canvas,
)
from diffusion_arc.inference import diagnose_exact_grids
from diffusion_arc.model import FluxSpatialShape, FluxVaeDecoder, FluxVaeEncoder

vae = AutoencoderKLFlux2.from_pretrained(
    LOCAL_VAE_PATH, torch_dtype=torch.float32, local_files_only=True,
    use_safetensors=True, low_cpu_mem_usage=True,
).eval().requires_grad_(False).to(DEVICE)
vae_encoder = FluxVaeEncoder(vae)
vae_decoder = FluxVaeDecoder(vae)
assert not vae.training and not any(parameter.requires_grad for parameter in vae.parameters())
assert next(vae.parameters()).dtype == torch.float32

print({
    'vae_class': type(vae).__name__,
    'vae_parameters': sum(parameter.numel() for parameter in vae.parameters()),
    'vae_sha256': vae_sha256,
    'task_sha256': task_sha256,
    'target_palette': TARGET_PALETTE_ID,
})


In [ ]:
runner_symbols = runpy.run_path(SOURCE_DIR / 'scripts' / 'experiments' / 'run_single_task_conditioning_ladder.py')
assert runner_symbols['TASK_ID'] == TASK_ID
CONDITION_SPECS = runner_symbols['CONDITION_SPECS']
PRODUCTION_SPATIAL_SHAPE = runner_symbols['SPATIAL_SHAPE']
assert (PRODUCTION_SPATIAL_SHAPE.height, PRODUCTION_SPATIAL_SHAPE.width) == (30, 30)
assert len(CONDITION_SPECS) == 17

task = load_training_task(ARC_TASK_PATH)
assert len(task.pairs) == 5
condition_metadata = []
target_grids = []
for condition in CONDITION_SPECS:
    key = condition.key()
    assert key.augmentation.pair_indices[-1] == 4
    assert set(key.augmentation.pair_indices[:-1]) == {0, 1, 2, 3}
    augmented = apply_training_augmentation(task, key.augmentation)
    target = augmented.pairs[-1].output_grid
    assert (len(target), len(target[0])) == (3, 3)
    target_grids.append(target)
    condition_metadata.append({
        'name': condition.name,
        'augmentation_key': str(key),
        'geometry': condition.geometry.value,
        'color_mapping': list(condition.color_mapping),
        'pair_indices': list(condition.pair_indices),
        'target_grid': [list(row) for row in target],
    })
target_grids = tuple(target_grids)
assert len({json.dumps(grid) for grid in target_grids}) == 17

display(pd.DataFrame([{
    'condition': item['name'], 'geometry': item['geometry'],
    'target': item['target_grid'],
} for item in condition_metadata]))


## 4. Production-faithful VAE round trip

The production path encodes batches of 16 and decodes batches of 8 on the top-left-aligned 30×30 canvas. The helper also retains the decoded RGB, classified cell canvas, and every mismatch so the failed condition is inspectable rather than reduced to `16/17`.


In [ ]:
def normalized_images_to_uint8(images: torch.Tensor) -> torch.Tensor:
    return images.detach().to(device='cpu', dtype=torch.float32).clamp(-1.0, 1.0).add(1.0).mul(127.5).round().to(torch.uint8)

def expected_class_canvas(grid, *, canvas_size: int, include_pad: bool) -> torch.Tensor:
    if include_pad:
        canvas = torch.full((canvas_size, canvas_size), PAD_CLASS, dtype=torch.int64)
        canvas[:len(grid), :len(grid[0])] = torch.tensor(grid, dtype=torch.int64)
        return canvas
    return torch.tensor(grid, dtype=torch.int64)

def cell_distance_details(rgb_image: torch.Tensor, expected: torch.Tensor, predicted: torch.Tensor, *, canvas_size: int, include_pad: bool) -> list[dict[str, object]]:
    cell_size = 16
    prototype_count = PAD_CLASS + 1 if include_pad else PAD_CLASS
    means = rgb_image.to(dtype=torch.float32).reshape(3, canvas_size, cell_size, canvas_size, cell_size).mean(dim=(2, 4)).permute(1, 2, 0)
    prototypes = TARGET_PALETTE_RGB[:prototype_count].to(dtype=torch.float32)
    distances = (means.unsqueeze(2) - prototypes.reshape(1, 1, -1, 3)).square().sum(dim=-1)
    best_values, best_indices = torch.topk(distances, k=2, largest=False, dim=-1)
    details = []
    for row, column in torch.nonzero(expected != predicted, as_tuple=False).tolist():
        expected_class = int(expected[row, column])
        details.append({
            'row': row,
            'column': column,
            'expected_class': expected_class,
            'predicted_class': int(predicted[row, column]),
            'cell_mean_rgb': [round(float(value), 3) for value in means[row, column].tolist()],
            'expected_distance_squared': round(float(distances[row, column, expected_class]), 4),
            'nearest_class': int(best_indices[row, column, 0]),
            'second_nearest_class': int(best_indices[row, column, 1]),
            'nearest_distance_squared': round(float(best_values[row, column, 0]), 4),
            'nearest_margin_squared': round(float(best_values[row, column, 1] - best_values[row, column, 0]), 4),
        })
    return details

def run_roundtrip(*, label: str, spatial_shape: FluxSpatialShape, include_pad: bool, encode_batch_size: int, decode_batch_size: int) -> dict[str, object]:
    canvas_size = spatial_shape.height
    assert spatial_shape.height == spatial_shape.width
    input_images = [render_grid_canvas(grid, canvas_size=canvas_size) for grid in target_grids]
    encoded = []
    torch.cuda.synchronize(DEVICE)
    started = time.perf_counter()
    for start in range(0, len(input_images), encode_batch_size):
        encoded.append(vae_encoder(torch.stack(input_images[start:start + encode_batch_size]), spatial_shape=spatial_shape))
    latents = torch.cat(encoded, dim=0)
    decoded_batches = []
    for start in range(0, len(target_grids), decode_batch_size):
        decoded_batches.append(vae_decoder(latents[start:start + decode_batch_size], spatial_shape=spatial_shape))
    decoded_images = torch.cat(decoded_batches, dim=0)
    torch.cuda.synchronize(DEVICE)
    elapsed_seconds = time.perf_counter() - started
    input_uint8 = normalized_images_to_uint8(torch.stack(input_images))
    decoded_uint8 = normalized_images_to_uint8(decoded_images)
    classes = classify_arc_cells(decoded_uint8, canvas_size=canvas_size, include_pad=include_pad)
    predictions = decode_arc_grids(classes, canvas_size=canvas_size) if include_pad else decode_fixed_arc_grids(classes, canvas_size=canvas_size)
    diagnostics = diagnose_exact_grids(predictions, target_grids)
    per_condition = []
    for index, (metadata, expected_grid, prediction, diagnostic) in enumerate(zip(condition_metadata, target_grids, predictions, diagnostics, strict=True)):
        expected_canvas = expected_class_canvas(expected_grid, canvas_size=canvas_size, include_pad=include_pad)
        predicted_canvas = classes[index]
        mismatch_details = cell_distance_details(decoded_uint8[index], expected_canvas, predicted_canvas, canvas_size=canvas_size, include_pad=include_pad)
        per_condition.append({
            **metadata,
            'exact_grid': diagnostic.exact_grid,
            'prediction_valid': diagnostic.prediction_valid,
            'invalid_reason': diagnostic.invalid_reason,
            'expected_shape': list(diagnostic.expected_shape),
            'predicted_shape': None if diagnostic.predicted_shape is None else list(diagnostic.predicted_shape),
            'grid_cell_accuracy': diagnostic.cell_accuracy,
            'decoded_grid': None if prediction.grid is None else [list(row) for row in prediction.grid],
            'full_canvas_mismatch_count': len(mismatch_details),
            'mismatches': mismatch_details,
        })
    exact_count = sum(item['exact_grid'] for item in per_condition)
    return {
        'label': label,
        'spatial_shape': [spatial_shape.height, spatial_shape.width],
        'include_pad': include_pad,
        'encode_batch_size': encode_batch_size,
        'decode_batch_size': decode_batch_size,
        'elapsed_seconds': elapsed_seconds,
        'exact_grids': exact_count,
        'total_grids': len(per_condition),
        'failed_conditions': [item['name'] for item in per_condition if not item['exact_grid']],
        'per_condition': per_condition,
        'latents': latents.detach().to(device='cpu', dtype=torch.float32),
        'input_images': input_uint8,
        'decoded_images': decoded_uint8,
        'classes': classes.detach().to(device='cpu', dtype=torch.int64),
    }


## 5. Separate padding from batching

The baseline exactly mirrors the failed runner: encode 16 then 1; decode 8, 8, then 1. The two alternative batchings isolate a hidden batch-dependence. The compact variant uses the same 17 transformed targets at 3×3 with no PAD boundary, matching the earlier spatial-ablation representation.


In [ ]:
def compare_roundtrips(reference: dict[str, object], candidate: dict[str, object]) -> dict[str, object]:
    reference_latents = reference['latents']
    candidate_latents = candidate['latents']
    reference_images = reference['decoded_images']
    candidate_images = candidate['decoded_images']
    reference_classes = reference['classes']
    candidate_classes = candidate['classes']
    return {
        'candidate': candidate['label'],
        'latent_max_abs_difference': float((reference_latents - candidate_latents).abs().max().item()),
        'decoded_rgb_max_abs_difference': int((reference_images.to(torch.int16) - candidate_images.to(torch.int16)).abs().max().item()),
        'classified_canvases_identical': bool(torch.equal(reference_classes, candidate_classes)),
        'failed_conditions_match': reference['failed_conditions'] == candidate['failed_conditions'],
    }

production = run_roundtrip(label='production-30x30-encode16-decode8', spatial_shape=PRODUCTION_SPATIAL_SHAPE, include_pad=True, encode_batch_size=16, decode_batch_size=8)
production_repeat = run_roundtrip(label='production-repeat', spatial_shape=PRODUCTION_SPATIAL_SHAPE, include_pad=True, encode_batch_size=16, decode_batch_size=8)
single_item = run_roundtrip(label='30x30-encode1-decode1', spatial_shape=PRODUCTION_SPATIAL_SHAPE, include_pad=True, encode_batch_size=1, decode_batch_size=1)
all_at_once = run_roundtrip(label='30x30-encode17-decode17', spatial_shape=PRODUCTION_SPATIAL_SHAPE, include_pad=True, encode_batch_size=17, decode_batch_size=17)
compact_shape = FluxSpatialShape(height=3, width=3)
compact = run_roundtrip(label='compact-3x3-encode16-decode8', spatial_shape=compact_shape, include_pad=False, encode_batch_size=16, decode_batch_size=8)

batching_checks = [
    compare_roundtrips(production, production_repeat),
    compare_roundtrips(production, single_item),
    compare_roundtrips(production, all_at_once),
]
summary_rows = [
    {key: value for key, value in run.items() if key in {'label', 'spatial_shape', 'include_pad', 'encode_batch_size', 'decode_batch_size', 'elapsed_seconds', 'exact_grids', 'total_grids', 'failed_conditions'}}
    for run in (production, production_repeat, single_item, all_at_once, compact)
]
display(pd.DataFrame(summary_rows))
display(pd.DataFrame(batching_checks))

compact_by_name = {item['name']: item for item in compact['per_condition']}
production_table = [{
    'condition': item['name'],
    'production_exact': item['exact_grid'],
    'production_shape': item['predicted_shape'],
    'production_cell_accuracy': item['grid_cell_accuracy'],
    'production_mismatches': item['full_canvas_mismatch_count'],
    'compact_exact': compact_by_name[item['name']]['exact_grid'],
    'compact_mismatches': compact_by_name[item['name']]['full_canvas_mismatch_count'],
} for item in production['per_condition']]
display(pd.DataFrame(production_table))


## 6. Visualize every target and save the durable diagnosis

Each row shows the expected 3×3 classes, the padded VAE reconstruction classified over a 6×6 top-left crop, the decoded RGB crop, and the compact reconstruction. The report contains the full mismatch list and nearest-palette distances.


In [ ]:
from PIL import Image, ImageDraw

def class_tile(class_canvas: torch.Tensor, *, cells: int, scale: int = 24) -> Image.Image:
    values = class_canvas[:cells, :cells].numpy()
    palette = TARGET_PALETTE_RGB.numpy()
    rgb = palette[values]
    image = Image.fromarray(rgb.astype(np.uint8), mode='RGB').resize((cells * scale, cells * scale), Image.Resampling.NEAREST)
    draw = ImageDraw.Draw(image)
    for coordinate in range(cells + 1):
        offset = coordinate * scale
        draw.line((offset, 0, offset, cells * scale), fill='black', width=1)
        draw.line((0, offset, cells * scale, offset), fill='black', width=1)
    return image

def rgb_crop_tile(rgb_image: torch.Tensor, *, cells: int, scale: int = 24) -> Image.Image:
    crop = rgb_image[:, :cells * 16, :cells * 16].permute(1, 2, 0).numpy()
    return Image.fromarray(crop.astype(np.uint8), mode='RGB').resize((cells * scale, cells * scale), Image.Resampling.NEAREST)

def labelled_tile(image: Image.Image, label: str, *, width: int = 170, height: int = 170) -> Image.Image:
    panel = Image.new('RGB', (width, height), 'white')
    draw = ImageDraw.Draw(panel)
    draw.text((4, 4), label, fill='black')
    image.thumbnail((width - 8, height - 28), Image.Resampling.NEAREST)
    panel.paste(image, ((width - image.width) // 2, 24 + (height - 28 - image.height) // 2))
    return panel

contact_rows = []
for index, result in enumerate(production['per_condition']):
    expected = expected_class_canvas(target_grids[index], canvas_size=3, include_pad=False)
    compact_classes = compact['classes'][index]
    panels = [
        labelled_tile(class_tile(expected, cells=3), 'expected 3x3'),
        labelled_tile(class_tile(production['classes'][index], cells=6), 'padded classes 6x6'),
        labelled_tile(rgb_crop_tile(production['decoded_images'][index], cells=6), 'padded RGB 6x6'),
        labelled_tile(class_tile(compact_classes, cells=3), 'compact classes'),
    ]
    row = Image.new('RGB', (sum(panel.width for panel in panels) + 10 * (len(panels) - 1), 198), 'white')
    row_draw = ImageDraw.Draw(row)
    status = 'EXACT' if result['exact_grid'] else f"FAIL: {result['invalid_reason'] or 'grid mismatch'}"
    row_draw.text((4, 176), f"{result['name']} — {status}; mismatched canvas cells: {result['full_canvas_mismatch_count']}", fill='black')
    x = 0
    for panel in panels:
        row.paste(panel, (x, 0))
        x += panel.width + 10
    contact_rows.append(row)

contact_sheet = Image.new('RGB', (max(row.width for row in contact_rows), sum(row.height for row in contact_rows)), 'white')
y = 0
for row in contact_rows:
    contact_sheet.paste(row, (0, y))
    y += row.height
CONTACT_SHEET_PATH = OUTPUT_DIR / 'roundtrip-contact-sheet.png'
contact_sheet.save(CONTACT_SHEET_PATH)
display(contact_sheet)
print(f'Contact sheet: {CONTACT_SHEET_PATH}')


In [ ]:
def serializable_run(result: dict[str, object]) -> dict[str, object]:
    return {key: value for key, value in result.items() if key not in {'latents', 'input_images', 'decoded_images', 'classes'}}

def interpret_result() -> str:
    padded_exact = production['exact_grids']
    compact_exact = compact['exact_grids']
    batching_identical = all(item['classified_canvases_identical'] for item in batching_checks)
    if padded_exact == production['total_grids']:
        return 'The original 16/17 failure did not reproduce under the pinned source and artifact hashes.'
    if compact_exact > padded_exact and batching_identical:
        return 'The compact path recovers more exact grids while all padded batching modes agree: this supports a padding-sensitive VAE failure.'
    if compact_exact == padded_exact and batching_identical:
        return 'The same error remains without padding and batching is invariant: padding and batch execution alone do not explain it.'
    if not batching_identical:
        return 'At least one alternative batch shape changes decoded classes: inspect batching_checks before interpreting the VAE failure.'
    return 'The compact and padded paths differ, but the result needs the per-condition report for interpretation.'

REPORT = {
    'schema_version': 1,
    'experiment': 'conditioning-ladder-vae-roundtrip-diagnosis',
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'source': {
        'repository_url': REPOSITORY_URL,
        'requested_commit': SOURCE_REVISION,
        'resolved_commit': RESOLVED_SOURCE_COMMIT,
        'runner_path': str(SOURCE_DIR / 'scripts' / 'experiments' / 'run_single_task_conditioning_ladder.py'),
    },
    'artifacts': {
        'vae_path': str(LOCAL_VAE_PATH),
        'vae_sha256': vae_sha256,
        'task_path': str(ARC_TASK_PATH),
        'task_sha256': task_sha256,
        'target_palette_id': TARGET_PALETTE_ID,
    },
    'runtime': {
        'python': sys.version,
        'torch': torch.__version__,
        'cuda': torch.version.cuda,
        'gpu': torch.cuda.get_device_properties(DEVICE).name,
        'gpu_memory_GiB': torch.cuda.get_device_properties(DEVICE).total_memory / 2**30,
    },
    'conditions': condition_metadata,
    'roundtrips': {
        'production': serializable_run(production),
        'production_repeat': serializable_run(production_repeat),
        'single_item': serializable_run(single_item),
        'all_at_once': serializable_run(all_at_once),
        'compact': serializable_run(compact),
    },
    'batching_checks': batching_checks,
    'interpretation': interpret_result(),
    'artifacts_written': {'contact_sheet': str(CONTACT_SHEET_PATH)},
}
REPORT_PATH = OUTPUT_DIR / 'report.json'
REPORT_PATH.write_text(json.dumps(REPORT, indent=2, sort_keys=True) + '\n', encoding='utf-8')

print(json.dumps({
    'production_exact_grids': f"{production['exact_grids']}/{production['total_grids']}",
    'production_failed_conditions': production['failed_conditions'],
    'compact_exact_grids': f"{compact['exact_grids']}/{compact['total_grids']}",
    'batching_checks': batching_checks,
    'interpretation': REPORT['interpretation'],
    'report': str(REPORT_PATH),
}, indent=2))


## Interpretation boundary

- If production fails and compact succeeds with identical batching outputs, the observed error is evidence for a padding-sensitive VAE representation issue for this target.
- If both paths fail identically, the issue is specific to the colour/geometry/motif combination or to VAE reconstruction more generally, not padding alone.
- If batching changes the classified output, record that as a numerical/runtime sensitivity before altering the ladder.
- This notebook does not choose a new palette and does not change the training runner. It produces the evidence required to make that later decision explicitly.
